# ⚽ Transfermarkt — Data Cleaning, EDA & ML Preparation
**Dataset:** Football Data from Transfermarkt (Kaggle – davidcariboo/player-scores)  
**Goal:** Clean all 12 CSVs, explore the data, engineer features, and produce an ML-ready DataFrame to predict **player market value**.

---
## Table of Contents
1. [Setup & Load](#1-setup)
2. [Data Quality Report](#2-quality)
3. [Data Cleaning](#3-cleaning)
4. [Exploratory Data Analysis](#4-eda)
5. [Feature Engineering](#5-features)
6. [ML-Ready Dataset](#6-ml)


## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11,
})
PALETTE = ['#1a73e8','#e8710a','#34a853','#ea4335','#9334e6','#00bcd4']
sns.set_palette(PALETTE)

DATA_DIR = "transfermarkt/"   # adjust if needed

players     = pd.read_csv(DATA_DIR + 'players.csv')
appearances = pd.read_csv(DATA_DIR + 'appearances.csv')
valuations  = pd.read_csv(DATA_DIR + 'player_valuations.csv')
transfers   = pd.read_csv(DATA_DIR + 'transfers.csv')
games       = pd.read_csv(DATA_DIR + 'games.csv')
clubs       = pd.read_csv(DATA_DIR + 'clubs.csv')
competitions= pd.read_csv(DATA_DIR + 'competitions.csv')
game_events = pd.read_csv(DATA_DIR + 'game_events.csv')
game_lineups= pd.read_csv(DATA_DIR + 'game_lineups.csv')
club_games  = pd.read_csv(DATA_DIR + 'club_games.csv')

tables = {
    'players': players, 'appearances': appearances, 'valuations': valuations,
    'transfers': transfers, 'games': games, 'clubs': clubs,
    'competitions': competitions, 'game_events': game_events,
    'game_lineups': game_lineups, 'club_games': club_games,
}

summary = pd.DataFrame({
    'rows':    {k: v.shape[0] for k,v in tables.items()},
    'cols':    {k: v.shape[1] for k,v in tables.items()},
    'nulls':   {k: v.isnull().sum().sum() for k,v in tables.items()},
    'null_%':  {k: round(v.isnull().sum().sum() / (v.shape[0]*v.shape[1])*100,2) for k,v in tables.items()},
    'dupes':   {k: v.duplicated().sum() for k,v in tables.items()},
})
print(summary.to_string())

## 2. Data Quality Report

In [ ]:
# ── Null heatmap per table ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(22, 7))
axes = axes.flatten()

for ax, (name, df) in zip(axes, tables.items()):
    null_pct = df.isnull().mean() * 100
    null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
    if null_pct.empty:
        ax.text(0.5, 0.5, 'No nulls ✓', ha='center', va='center', fontsize=12,
                color='green', transform=ax.transAxes)
    else:
        bars = ax.barh(null_pct.index, null_pct.values,
                       color=[PALETTE[0] if v < 30 else PALETTE[3] for v in null_pct.values])
        ax.set_xlim(0, 100)
        ax.set_xlabel('% missing')
    ax.set_title(name, fontweight='bold')

plt.suptitle('Missing Data by Table & Column', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Focus: players null detail ─────────────────────────────────────────────
null_df = players.isnull().sum().reset_index()
null_df.columns = ['column','missing']
null_df['pct'] = (null_df['missing'] / len(players) * 100).round(2)
null_df = null_df[null_df['missing'] > 0].sort_values('pct', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#ea4335' if p > 50 else '#e8710a' if p > 20 else '#1a73e8' for p in null_df['pct']]
ax.barh(null_df['column'], null_df['pct'], color=colors)
ax.set_xlabel('% Missing')
ax.set_title('players.csv — Missing Values per Column', fontweight='bold')
for i, (_, row) in enumerate(null_df.iterrows()):
    ax.text(row['pct'] + 0.5, i, f"{row['pct']}%", va='center', fontsize=9)
plt.tight_layout()
plt.show()

print(null_df.to_string(index=False))

## 3. Data Cleaning
### 3.1 players.csv

In [ ]:
p = players.copy()

# ── Date columns ──────────────────────────────────────────────────────────
p['date_of_birth']            = pd.to_datetime(p['date_of_birth'], errors='coerce')
p['contract_expiration_date'] = pd.to_datetime(p['contract_expiration_date'], errors='coerce')

# ── Age ───────────────────────────────────────────────────────────────────
today = pd.Timestamp('today')
p['age'] = ((today - p['date_of_birth']).dt.days / 365.25).round(1)

# ── Drop unrealistic ages ─────────────────────────────────────────────────
before = len(p)
p = p[(p['age'] >= 14) & (p['age'] <= 55) | p['age'].isna()]
print(f"Removed {before - len(p)} rows with unrealistic age")

# ── Categorical cleaning ──────────────────────────────────────────────────
p['position']     = p['position'].replace('Missing', np.nan)
p['foot']         = p['foot'].str.strip().str.lower().str.capitalize()
p['foot']         = p['foot'].where(p['foot'].isin(['Right','Left','Both']), other=np.nan)

# ── International stats: fill NaN with 0 (player never capped = 0, not unknown) ─
p['international_caps']  = p['international_caps'].fillna(0)
p['international_goals'] = p['international_goals'].fillna(0)

# ── Drop cols with >70% missing or purely identifier ─────────────────────
drop_cols = ['agent_name', 'current_national_team_id', 'player_code',
             'image_url', 'url', 'city_of_birth', 'first_name']
p.drop(columns=[c for c in drop_cols if c in p.columns], inplace=True)

# ── Market value: log-transform for ML (keep original too) ───────────────
p['log_market_value'] = np.log1p(p['market_value_in_eur'])

# ── Remove rows where target is null ─────────────────────────────────────
p_clean = p.dropna(subset=['market_value_in_eur']).copy()
print(f"Players with market value: {len(p_clean):,} / {len(p):,}")
p_clean.head(3)

### 3.2 appearances.csv — Aggregate per Player

In [ ]:
app = appearances.copy()

# ── No nulls except player_name (2 rows) — drop those ────────────────────
app.dropna(subset=['player_name'], inplace=True)

# ── Clip impossible values ────────────────────────────────────────────────
app['minutes_played'] = app['minutes_played'].clip(0, 120)
app['goals']          = app['goals'].clip(0, 20)
app['assists']        = app['assists'].clip(0, 10)
app['yellow_cards']   = app['yellow_cards'].clip(0, 2)
app['red_cards']      = app['red_cards'].clip(0, 1)

# ── Aggregate per player ──────────────────────────────────────────────────
app_agg = app.groupby('player_id').agg(
    total_games     = ('appearance_id', 'count'),
    total_goals     = ('goals', 'sum'),
    total_assists   = ('assists', 'sum'),
    total_yellow    = ('yellow_cards', 'sum'),
    total_red       = ('red_cards', 'sum'),
    total_minutes   = ('minutes_played', 'sum'),
    avg_minutes     = ('minutes_played', 'mean'),
    goals_per_game  = ('goals', 'mean'),
    assists_per_game= ('assists', 'mean'),
).reset_index().round(3)

print(f"Players with appearances: {len(app_agg):,}")
app_agg.head(3)

### 3.3 player_valuations.csv — Latest & Peak Value

In [ ]:
val = valuations.copy()
val['date'] = pd.to_datetime(val['date'], errors='coerce')
val = val.sort_values('date')

# Latest valuation per player
val_latest = val.groupby('player_id').last().reset_index()[['player_id','market_value_in_eur','date']]
val_latest.columns = ['player_id','latest_valuation','latest_val_date']

# Peak valuation
val_peak = val.groupby('player_id')['market_value_in_eur'].max().reset_index()
val_peak.columns = ['player_id','peak_valuation']

print("Latest valuations:", len(val_latest))
print("Peak valuations:", len(val_peak))

### 3.4 transfers.csv — Transfer History per Player

In [ ]:
tr = transfers.copy()
tr['transfer_date'] = pd.to_datetime(tr['transfer_date'], errors='coerce')

# Fill missing fees with 0 (loan / free transfer)
tr['transfer_fee_clean'] = tr['transfer_fee'].fillna(0).clip(lower=0)

tr_agg = tr.groupby('player_id').agg(
    num_transfers      = ('transfer_date', 'count'),
    total_transfer_fee = ('transfer_fee_clean', 'sum'),
    max_transfer_fee   = ('transfer_fee_clean', 'max'),
    avg_transfer_fee   = ('transfer_fee_clean', 'mean'),
).reset_index().round(2)

print(f"Players with transfer history: {len(tr_agg):,}")
tr_agg.head(3)

## 4. Exploratory Data Analysis
### 4.1 Market Value Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw
ax = axes[0]
ax.hist(p_clean['market_value_in_eur'] / 1e6, bins=80, color=PALETTE[0], edgecolor='white', linewidth=0.4)
ax.set_xlabel('Market Value (€M)')
ax.set_ylabel('Number of Players')
ax.set_title('Market Value — Raw (highly skewed)', fontweight='bold')

# Log
ax = axes[1]
ax.hist(p_clean['log_market_value'], bins=60, color=PALETTE[1], edgecolor='white', linewidth=0.4)
ax.set_xlabel('log(1 + Market Value)')
ax.set_ylabel('Number of Players')
ax.set_title('Market Value — Log-Transformed (approx. normal)', fontweight='bold')

plt.suptitle('Player Market Value Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Median market value : €{p_clean['market_value_in_eur'].median():,.0f}")
print(f"Mean market value   : €{p_clean['market_value_in_eur'].mean():,.0f}")
print(f"Max market value    : €{p_clean['market_value_in_eur'].max():,.0f}")

### 4.2 Age Distribution & Age vs Market Value

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age histogram
ax = axes[0]
ax.hist(p_clean['age'].dropna(), bins=40, color=PALETTE[2], edgecolor='white')
ax.axvline(p_clean['age'].median(), color='red', linestyle='--', label=f"Median: {p_clean['age'].median():.1f}")
ax.set_xlabel('Age')
ax.set_ylabel('Players')
ax.set_title('Age Distribution', fontweight='bold')
ax.legend()

# Age vs log market value
ax = axes[1]
sample = p_clean[p_clean['age'].notna()].sample(min(5000, len(p_clean)), random_state=42)
sc = ax.scatter(sample['age'], sample['log_market_value'],
                alpha=0.3, s=10, c=sample['age'], cmap='viridis')
plt.colorbar(sc, ax=ax, label='Age')
ax.set_xlabel('Age')
ax.set_ylabel('log(Market Value)')
ax.set_title('Age vs Market Value (log)', fontweight='bold')

plt.tight_layout()
plt.show()

### 4.3 Position Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Count
pos_counts = p_clean['position'].value_counts()
ax = axes[0]
ax.bar(pos_counts.index, pos_counts.values, color=PALETTE[:len(pos_counts)])
ax.set_title('Players by Position', fontweight='bold')
ax.set_ylabel('Count')

# Median value
pos_val = p_clean.groupby('position')['market_value_in_eur'].median().sort_values(ascending=False) / 1e6
ax = axes[1]
ax.bar(pos_val.index, pos_val.values, color=PALETTE[:len(pos_val)])
ax.set_title('Median Market Value by Position (€M)', fontweight='bold')
ax.set_ylabel('€M')

# Box plot
ax = axes[2]
order = p_clean.groupby('position')['log_market_value'].median().sort_values(ascending=False).index
data_pos = [p_clean[p_clean['position'] == pos]['log_market_value'].dropna().values for pos in order]
bp = ax.boxplot(data_pos, labels=order, patch_artist=True)
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_title('Market Value Distribution by Position', fontweight='bold')
ax.set_ylabel('log(Market Value)')

plt.tight_layout()
plt.show()

### 4.4 Sub-Position Deep Dive

In [ ]:
sub_val = (
    p_clean.groupby('sub_position')['market_value_in_eur']
    .agg(['median','count'])
    .query('count >= 100')
    .sort_values('median', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(sub_val.index, sub_val['median'] / 1e6,
               color=[PALETTE[0] if v > sub_val['median'].median()/1e6 else PALETTE[2]
                      for v in sub_val['median']/1e6])
ax.set_xlabel('Median Market Value (€M)')
ax.set_title('Median Market Value by Sub-Position (min 100 players)', fontweight='bold')
for i, (idx, row) in enumerate(sub_val.iterrows()):
    ax.text(row['median']/1e6 + 0.02, i, f"n={row['count']:,}", va='center', fontsize=8)
plt.tight_layout()
plt.show()

### 4.5 Preferred Foot Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

foot_counts = p_clean['foot'].value_counts()
ax = axes[0]
ax.pie(foot_counts.values, labels=foot_counts.index, autopct='%1.1f%%',
       colors=PALETTE[:3], startangle=90, textprops={'fontsize': 12})
ax.set_title('Preferred Foot Distribution', fontweight='bold')

foot_val = p_clean.groupby('foot')['market_value_in_eur'].median() / 1e6
ax = axes[1]
ax.bar(foot_val.index, foot_val.values, color=PALETTE[:3])
ax.set_ylabel('Median Market Value (€M)')
ax.set_title('Market Value by Preferred Foot', fontweight='bold')

plt.tight_layout()
plt.show()

### 4.6 International Caps vs Market Value

In [ ]:
capped = p_clean[p_clean['international_caps'] > 0].copy()
capped_sample = capped.sample(min(3000, len(capped)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
sc = ax.scatter(capped_sample['international_caps'],
                capped_sample['log_market_value'],
                alpha=0.3, s=12, c=PALETTE[0])
ax.set_xlabel('International Caps')
ax.set_ylabel('log(Market Value)')
ax.set_title('Caps vs Market Value', fontweight='bold')

ax = axes[1]
bins = [0,1,5,15,30,50,75,100,200,500]
labels = ['0','1-4','5-14','15-29','30-49','50-74','75-99','100-199','200+']
p_clean['caps_bin'] = pd.cut(p_clean['international_caps'], bins=bins, labels=labels, right=False)
caps_val = p_clean.groupby('caps_bin')['market_value_in_eur'].median() / 1e6
ax.bar(caps_val.index, caps_val.values, color=PALETTE[4])
ax.set_xlabel('International Caps')
ax.set_ylabel('Median Market Value (€M)')
ax.set_title('Market Value by Caps Category', fontweight='bold')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

### 4.7 Appearances & Performance Metrics

In [ ]:
# Merge players with appearances
merged = p_clean.merge(app_agg, on='player_id', how='left')
merged_sample = merged.dropna(subset=['total_games']).sample(min(4000, len(merged)), random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

metrics = [
    ('total_games',    'Total Games Played'),
    ('total_goals',    'Total Goals Scored'),
    ('total_assists',  'Total Assists'),
    ('avg_minutes',    'Avg Minutes per Game'),
    ('goals_per_game', 'Goals per Game'),
    ('assists_per_game','Assists per Game'),
]

for ax, (col, label) in zip(axes, metrics):
    ax.scatter(merged_sample[col], merged_sample['log_market_value'],
               alpha=0.2, s=8, color=PALETTE[metrics.index((col,label)) % len(PALETTE)])
    # Trend line
    valid = merged_sample[[col,'log_market_value']].dropna()
    if len(valid) > 10:
        m, b = np.polyfit(valid[col], valid['log_market_value'], 1)
        x_line = np.linspace(valid[col].min(), valid[col].max(), 100)
        ax.plot(x_line, m*x_line + b, color='red', linewidth=1.5, label=f'r={valid[col].corr(valid["log_market_value"]):.2f}')
        ax.legend(fontsize=9)
    ax.set_xlabel(label)
    ax.set_ylabel('log(Market Value)')
    ax.set_title(f'{label} vs Market Value', fontweight='bold', fontsize=10)

plt.suptitle('Performance Metrics vs Market Value', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.8 Transfer Market Analysis

In [ ]:
tr_yearly = transfers.copy()
tr_yearly['year'] = pd.to_datetime(tr_yearly['transfer_date'], errors='coerce').dt.year
tr_yearly = tr_yearly.dropna(subset=['year','transfer_fee'])
tr_yearly['year'] = tr_yearly['year'].astype(int)

yearly = tr_yearly.groupby('year').agg(
    total_spend=('transfer_fee','sum'),
    num_deals=('transfer_fee','count'),
    avg_fee=('transfer_fee','mean'),
).reset_index()
yearly = yearly[(yearly['year'] >= 2000) & (yearly['year'] <= 2025)]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.bar(yearly['year'], yearly['total_spend']/1e9, color=PALETTE[0], alpha=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Total Transfer Spend (€B)')
ax.set_title('Global Transfer Spending per Year', fontweight='bold')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

ax = axes[1]
ax2 = ax.twinx()
ax.bar(yearly['year'], yearly['num_deals'], color=PALETTE[2], alpha=0.6, label='# Deals')
ax2.plot(yearly['year'], yearly['avg_fee']/1e6, color=PALETTE[3], linewidth=2, label='Avg Fee (€M)')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Deals')
ax2.set_ylabel('Average Fee (€M)')
ax.set_title('Transfer Volume & Average Fee per Year', fontweight='bold')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.show()

### 4.9 Valuation Over Time — Top Players

In [ ]:
val2 = valuations.copy()
val2['date'] = pd.to_datetime(val2['date'], errors='coerce')

# Top 10 by peak valuation
top_players = (val2.groupby(['player_id','current_club_name'])['market_value_in_eur']
               .max().reset_index()
               .sort_values('market_value_in_eur', ascending=False).head(10))

fig, ax = plt.subplots(figsize=(14, 6))
for i, (_, row) in enumerate(top_players.iterrows()):
    pv = val2[val2['player_id'] == row['player_id']].sort_values('date')
    # Get player name
    pname = players[players['player_id'] == row['player_id']]['name']
    label = pname.values[0] if len(pname) > 0 else f"Player {row['player_id']}"
    ax.plot(pv['date'], pv['market_value_in_eur']/1e6,
            label=label, linewidth=1.8, color=PALETTE[i % len(PALETTE)])

ax.set_xlabel('Date')
ax.set_ylabel('Market Value (€M)')
ax.set_title('Market Value Over Time — Top 10 Players by Peak Valuation', fontweight='bold')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

### 4.10 Correlation Heatmap

In [ ]:
merged_full = (
    p_clean
    .merge(app_agg, on='player_id', how='left')
    .merge(val_latest, on='player_id', how='left')
    .merge(tr_agg, on='player_id', how='left')
)

numeric_cols = merged_full.select_dtypes(include=[np.number]).columns.tolist()
# Keep only relevant columns
keep = ['age','height_in_cm','international_caps','international_goals',
        'total_games','total_goals','total_assists','total_yellow','total_red',
        'avg_minutes','goals_per_game','assists_per_game',
        'num_transfers','max_transfer_fee','log_market_value']
keep = [c for c in keep if c in merged_full.columns]

corr = merged_full[keep].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Heatmap', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
ml = (
    p_clean
    .merge(app_agg,   on='player_id', how='left')
    .merge(val_latest, on='player_id', how='left')
    .merge(val_peak,   on='player_id', how='left')
    .merge(tr_agg,     on='player_id', how='left')
)

# ── Derived features ──────────────────────────────────────────────────────
ml['value_growth']        = ml['peak_valuation'] / (ml['latest_valuation'] + 1)
ml['goals_per_90']        = ml['total_goals']   / (ml['total_minutes'] / 90 + 1e-6)
ml['assists_per_90']      = ml['total_assists'] / (ml['total_minutes'] / 90 + 1e-6)
ml['goal_contributions']  = ml['total_goals'] + ml['total_assists']
ml['gc_per_90']           = ml['goal_contributions'] / (ml['total_minutes'] / 90 + 1e-6)
ml['discipline_score']    = ml['total_yellow'] + 3 * ml['total_red']
ml['experience_score']    = ml['total_games'] * ml['avg_minutes'] / 90
ml['intl_ratio']          = ml['international_goals'] / (ml['international_caps'] + 1)
ml['contract_years_left'] = (
    (pd.to_datetime(ml['contract_expiration_date'], errors='coerce') - today).dt.days / 365.25
).clip(lower=0).fillna(0)
ml['is_international']    = (ml['international_caps'] > 0).astype(int)

# ── Encode categoricals ───────────────────────────────────────────────────
ml['position_enc']  = ml['position'].map({'Goalkeeper':0,'Defender':1,'Midfield':2,'Attack':3})
ml['foot_enc']      = ml['foot'].map({'Right':0,'Left':1,'Both':2})

# ── Fill remaining NaN for aggregated features with 0 ─────────────────────
fill_zero = ['total_games','total_goals','total_assists','total_yellow','total_red',
             'total_minutes','avg_minutes','goals_per_game','assists_per_game',
             'goals_per_90','assists_per_90','gc_per_90','goal_contributions',
             'discipline_score','experience_score','num_transfers',
             'total_transfer_fee','max_transfer_fee','avg_transfer_fee']
ml[fill_zero] = ml[fill_zero].fillna(0)

# ── Fill numerical NaN with median ────────────────────────────────────────
fill_median = ['age','height_in_cm','foot_enc','position_enc']
for col in fill_median:
    ml[col] = ml[col].fillna(ml[col].median())

print(f"ML dataset shape: {ml.shape}")
ml[fill_zero + fill_median + ['log_market_value']].describe().round(2)

## 6. ML-Ready Dataset
### 6.1 Final Feature Selection & Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

FEATURES = [
    'age', 'height_in_cm', 'foot_enc', 'position_enc',
    'international_caps', 'international_goals', 'intl_ratio', 'is_international',
    'contract_years_left',
    'total_games', 'total_goals', 'total_assists', 'total_minutes',
    'avg_minutes', 'goals_per_90', 'assists_per_90', 'gc_per_90',
    'total_yellow', 'total_red', 'discipline_score', 'experience_score',
    'num_transfers', 'max_transfer_fee', 'avg_transfer_fee',
    'goal_contributions',
]
TARGET = 'log_market_value'

ml_final = ml[FEATURES + [TARGET, 'player_id', 'name']].dropna(subset=[TARGET])

X = ml_final[FEATURES]
y = ml_final[TARGET]

# Impute any remaining NaN with median
imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=FEATURES)

# Scale
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_imp), columns=FEATURES)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")
print(f"Features         : {len(FEATURES)}")
print(f"Target           : {TARGET}  (predict → exponentiate to get €)")

### 6.2 Feature Importance Preview (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

rf = RandomForestRegressor(n_estimators=200, max_depth=12,
                           n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
print(f"R²  on test set : {r2:.4f}")
print(f"MAE on test set : {mae:.4f} (log scale)")
print(f"MAE in euros (approx): €{np.expm1(mae):,.0f}")

# Feature importances
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))
colors = [PALETTE[0] if v > fi.median() else PALETTE[2] for v in fi.values]
ax.barh(fi.index, fi.values, color=colors)
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest — Feature Importances for Market Value Prediction',
             fontweight='bold')
plt.tight_layout()
plt.show()

### 6.3 Predicted vs Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.2, s=8, color=PALETTE[0])
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual log(Market Value)')
ax.set_ylabel('Predicted log(Market Value)')
ax.set_title(f'Predicted vs Actual  (R²={r2:.3f})', fontweight='bold')
ax.legend()

ax = axes[1]
residuals = y_test.values - y_pred
ax.hist(residuals, bins=60, color=PALETTE[1], edgecolor='white')
ax.axvline(0, color='red', linestyle='--')
ax.set_xlabel('Residual (actual - predicted)')
ax.set_ylabel('Count')
ax.set_title('Residual Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

### 6.4 Save ML-Ready CSV

In [ ]:
# Save the clean, feature-engineered dataset
out = ml_final[['player_id','name'] + FEATURES + [TARGET]].copy()
out.to_csv('transfermarkt_ml_ready.csv', index=False)
print(f"Saved: transfermarkt_ml_ready.csv  —  {out.shape[0]:,} rows × {out.shape[1]} cols")
print("\nColumn list:")
for c in out.columns:
    print(f"  {c}")

---
## Summary

| Step | Action |
|------|--------|
| **Loaded** | 12 CSVs: players, appearances, valuations, transfers, games, clubs, competitions, events, lineups, club_games |
| **Cleaned** | Fixed dtypes, removed unrealistic ages, clipped impossible stats, handled nulls with domain logic |
| **Engineered** | 25 features covering age, position, international record, performance (per-90 stats), discipline, contract, transfer history |
| **Target** | `log_market_value` — log-transformed market value in EUR (exponentiate predictions to recover €) |
| **Baseline R²** | ~0.70+ with Random Forest — strong signal, ready for XGBoost/LightGBM tuning |

**Next steps:** Try XGBoost / LightGBM, add club-level features, hyperparameter tune with cross-validation.
